# Categorical Encoding

**Proyecto:** Predicción de subempleo por insuficiencia de horas – EPEN 2024 (INEI Perú)

## Objetivo

Este notebook transforma las variables categóricas del dataset en un formato numérico utilizable por modelos de Machine Learning.  
Se trabaja sobre el dataset filtrado con el target ya definido (`epen_target_defined.csv`) y se aplica:

- Recodificación binaria (1/2 → 1/0) para variables dicotómicas.
- Conservación ordinal para nivel educativo y tamaño de empresa.
- One-Hot Encoding para variables nominales.

**Restricciones:** No se realiza train/test split, balanceo de clases ni entrenamiento de modelos.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ─── Rutas ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_PROC    = PROJECT_DIR / 'data' / 'processed'
INPUT_FILE   = DATA_PROC / 'epen_target_defined.csv'

print(f'Archivo de entrada: {INPUT_FILE}')
print(f'Existe: {INPUT_FILE.exists()}')

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        'No se encontró epen_target_defined.csv. '
        'Ejecuta primero el notebook target_definition.ipynb.'
    )

df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig', low_memory=False)
print(f'\nDataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

Archivo de entrada: g:\Mi unidad\UP - Ingeniería de la información\Semestre IX\Machine Learning\TrabajoFinal\clonProyecto\ML_PROYECTO_26_1\ml_project\data\processed\epen_target_defined.csv
Existe: True

Dataset cargado: 24,054 filas × 136 columnas


,ANIO,MES,CONGLOMERADO,MUESTRA,SELVIV,HOGAR,REGION,LLAVE_PANEL,ESTRATO,C201,...,ingtrabw,RESIDENT,fa_ond24,fa_efm24,fa_amj24,fa_jas24,C208_num,OCUP300_num,P209H_num,target_subempleo_horas
0,2024,10,18117,2,61,1,1,2.024102e+17,1,1,...,1367,1,700.400712,NaN,NaN,NaN,60.0,1.0,2.0,0
1,2024,10,18117,2,61,1,1,2.024102e+17,1,2,...,1534,1,700.400712,NaN,NaN,NaN,49.0,1.0,1.0,1
2,2024,10,1823302,1,59,1,1,2.024102e+19,1,1,...,1400,1,896.469464,NaN,NaN,NaN,52.0,1.0,2.0,0


## 1. Revisión de columnas disponibles y target

In [2]:
print('Total de columnas:', df.shape[1])
print('Target presente:', 'target_subempleo_horas' in df.columns)
print('\nDistribución del target:')
print(df['target_subempleo_horas'].value_counts().rename({0: 'No subempleado (0)', 1: 'Subempleado (1)'}).to_string())

Total de columnas: 136
Target presente: True

Distribución del target:
target_subempleo_horas
No subempleado (0)    18064
Subempleado (1)        5990


## 2. Variables excluidas del encoding

### Variables con riesgo de data leakage
`P209H`, `C333` y `C334` están directamente relacionadas con la construcción del target:
- **P209H**: variable fuente del target (quería y podía trabajar más horas).
- **C333**: ¿quería trabajar más horas esta semana?
- **C334**: ¿estuvo disponible para trabajar más horas?

Incluirlas como predictoras causaría data leakage — el modelo aprendería la respuesta en lugar del patrón.

### Variables administrativas / identificadores
No aportan información predictiva: `ANIO`, `MES`, `CONGLOMERADO`, `MUESTRA`, `SELVIV`, `HOGAR`, `LLAVE_PANEL`, `C201`, `C300n`, `NROINF`, pesos muestrales.

In [3]:
leakage_variables = ['P209H', 'C333', 'C334']

id_variables = [
    'ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'LLAVE_PANEL',
    'C201', 'C300n', 'NROINF',
    'fa_ond24', 'fa_efm24', 'fa_amj24', 'fa_jas24',  # pesos muestrales
    'C208_num', 'OCUP300_num', 'P209H_num',           # columnas auxiliares de limpieza
    'REGION', 'OCUP300', 'RESIDENT',                   # usadas como filtro, no como predictores
]

print('Variables de leakage:', leakage_variables)
print('Variables administrativas a excluir (presentes en df):',
      [c for c in id_variables if c in df.columns])

Variables de leakage: ['P209H', 'C333', 'C334']
Variables administrativas a excluir (presentes en df): ['ANIO', 'MES', 'CONGLOMERADO', 'MUESTRA', 'SELVIV', 'HOGAR', 'LLAVE_PANEL', 'C201', 'C300n', 'NROINF', 'fa_ond24', 'fa_efm24', 'fa_amj24', 'fa_jas24', 'C208_num', 'OCUP300_num', 'P209H_num', 'REGION', 'OCUP300', 'RESIDENT']


## 3. Definición de variables candidatas por tipo

| Tipo | Variables | Tratamiento |
|---|---|---|
| **Numéricas** | C208, C318_T, whoraT, INGTOT, INGTOTP, ingtrabw | Se conservan tal cual |
| **Binarias (1/2→1/0)** | C207, C361_1, C361_5, C364_1, C364_2, C375_1–C375_6, C335 | Recodificación |
| **Ordinales** | C366 (educación), C317 (tamaño empresa) | Se conservan como numéricas ordinales |
| **Nominales (OHE)** | C203, C310, C311, C312, C313, SEGURO1, C376, C377 | One-Hot Encoding |

In [4]:
# Columnas numéricas a conservar directamente
NUM_VARS = ['C208', 'C318_T', 'whoraT', 'INGTOT', 'INGTOTP', 'ingtrabw']

# Variables binarias (1=Sí / 2=No → 1/0)
BIN_VARS = ['C207', 'C361_1', 'C361_5', 'C364_1', 'C364_2',
            'C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6', 'C335']

# Variables ordinales (se conservan como entero)
ORD_VARS = ['C366', 'C317']

# Variables nominales para OHE
OHE_VARS = ['C203', 'C310', 'C311', 'C312', 'C313', 'SEGURO1', 'C376', 'C377']

# Filtrar solo las que existen en el dataset
num_ok  = [c for c in NUM_VARS  if c in df.columns]
bin_ok  = [c for c in BIN_VARS  if c in df.columns]
ord_ok  = [c for c in ORD_VARS  if c in df.columns]
ohe_ok  = [c for c in OHE_VARS  if c in df.columns]

print(f'Numéricas presentes:  {num_ok}')
print(f'Binarias presentes:   {bin_ok}')
print(f'Ordinales presentes:  {ord_ok}')
print(f'Nominales presentes:  {ohe_ok}')

Numéricas presentes:  ['C208', 'C318_T', 'whoraT', 'INGTOT', 'INGTOTP', 'ingtrabw']
Binarias presentes:   ['C207', 'C361_1', 'C361_5', 'C364_1', 'C364_2', 'C375_1', 'C375_2', 'C375_3', 'C375_4', 'C375_5', 'C375_6', 'C335']
Ordinales presentes:  ['C366', 'C317']
Nominales presentes:  ['C203', 'C310', 'C311', 'C312', 'C313', 'SEGURO1', 'C376', 'C377']


## 4. Tratamiento previo de códigos especiales de missing

Antes de codificar, se reemplazan los códigos de no respuesta por `NaN` **solo en las columnas donde corresponde**, según el diccionario EPEN 2024:

- `C311`, `C312`, `C313`, `C335`: vienen como `str` con espacios en blanco → reemplazar por `NaN` y convertir a numérico.
- `C203`: código `9` = no respuesta.
- `C310`: código `9` = no especificado.
- Ingresos (`INGTOT`, `INGTOTP`, `ingtrabw`): código `999999` → `NaN`.
- `C318_T`: código `99` → `NaN`. `whoraT`: código `999` → `NaN`.

In [5]:
df_work = df.copy()

# Variables tipo str con espacios como missing (C311, C312, C313, C335)
str_vars = ['C311', 'C312', 'C313', 'C335']
for col in str_vars:
    if col in df_work.columns:
        df_work[col] = df_work[col].replace(r'^\s*$', np.nan, regex=True)
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce')

# C203: código 9 = no respuesta
if 'C203' in df_work.columns:
    df_work['C203'] = df_work['C203'].replace(9, np.nan)

# C310: código 9 = no especificado
if 'C310' in df_work.columns:
    df_work['C310'] = df_work['C310'].replace(9, np.nan)

# Ingresos: 999999 → NaN (por si no se aplicó antes)
for col in ['INGTOT', 'INGTOTP', 'ingtrabw']:
    if col in df_work.columns:
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce').replace(999999, np.nan)

# C318_T: 99 → NaN
if 'C318_T' in df_work.columns:
    df_work['C318_T'] = pd.to_numeric(df_work['C318_T'], errors='coerce').replace(99, np.nan)

# whoraT: 999 → NaN
if 'whoraT' in df_work.columns:
    df_work['whoraT'] = pd.to_numeric(df_work['whoraT'], errors='coerce').replace(999, np.nan)

print('Limpieza de códigos especiales aplicada.')
print('Nulos tras limpieza (variables clave):')
check_cols = ['C203', 'C310', 'C311', 'C312', 'C313', 'C335', 'C318_T', 'whoraT', 'INGTOT']
print(df_work[[c for c in check_cols if c in df_work.columns]].isna().sum().to_string())

Limpieza de códigos especiales aplicada.
Nulos tras limpieza (variables clave):
C203         72
C310         70
C311      10164
C312       3173
C313      11429
C335        744
C318_T        1
whoraT        0
INGTOT      774


## 5. Codificación binaria (1/2 → 1/0)

Las variables dicotómicas del EPEN codifican **1 = Sí** y **2 = No**. Se recodifican a **1 = Sí** y **0 = No** para que los modelos las interpreten correctamente como binarias.  
Se crean nuevas columnas con sufijo `_bin`, conservando las originales hasta la limpieza final.

In [6]:
bin_ok_current = [c for c in bin_ok if c in df_work.columns]

for col in bin_ok_current:
    new_col = f'{col}_bin'
    df_work[new_col] = df_work[col].map({1: 1, 2: 0})

print('Variables binarias recodificadas:')
for col in bin_ok_current:
    new_col = f'{col}_bin'
    vc = df_work[new_col].value_counts(dropna=False).sort_index()
    print(f'  {new_col}: {vc.to_dict()}')

Variables binarias recodificadas:
  C207_bin: {0: 11310, 1: 12744}
  C361_1_bin: {0: 13799, 1: 10255}
  C361_5_bin: {0: 12883, 1: 11171}
  C364_1_bin: {0: 14179, 1: 9875}
  C364_2_bin: {0: 20969, 1: 3085}
  C375_1_bin: {0: 23981, 1: 73}
  C375_2_bin: {0: 24040, 1: 14}
  C375_3_bin: {0: 24044, 1: 10}
  C375_4_bin: {0: 24020, 1: 34}
  C375_5_bin: {0: 24009, 1: 45}
  C375_6_bin: {0: 24012, 1: 42}
  C335_bin: {0.0: 20945, 1.0: 2365, nan: 744}


## 6. Variables ordinales

`C366` (nivel educativo) y `C317` (tamaño de empresa) tienen un **orden natural** implícito — a mayor código, mayor nivel educativo o mayor tamaño de empresa.  
Por eso se conservan como enteros ordinales en lugar de aplicarles One-Hot Encoding, que perdería esa información de orden.

In [7]:
if 'C366' in df_work.columns:
    df_work['nivel_educativo_ord'] = df_work['C366'].astype(float)
    print('C366 → nivel_educativo_ord')
    print('  Valores únicos:', sorted(df_work['nivel_educativo_ord'].dropna().unique().tolist()))

if 'C317' in df_work.columns:
    df_work['tamano_empresa_ord'] = df_work['C317'].astype(float)
    print('C317 → tamano_empresa_ord')
    print('  Valores únicos:', sorted(df_work['tamano_empresa_ord'].dropna().unique().tolist()))

C366 → nivel_educativo_ord
  Valores únicos: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0]
C317 → tamano_empresa_ord
  Valores únicos: [1.0, 2.0, 3.0, 4.0, 5.0]


## 7. One-Hot Encoding para variables nominales

Se aplica `pd.get_dummies` con `drop_first=True` (elimina la primera categoría para evitar multicolinealidad) a las variables nominales que no tienen orden natural.  
Se usan prefijos descriptivos para facilitar la interpretación de las columnas resultantes.

In [8]:
ohe_prefixes = {
    'C203':   'C203_parentesco',
    'C310':   'C310_cat_ocup',
    'C311':   'C311_tipo_entidad',
    'C312':   'C312_registro_sunat',
    'C313':   'C313_contabilidad',
    'SEGURO1':'seguro_salud',
    'C376':   'C376_lengua',
    'C377':   'C377_etnia',
}

ohe_ok_current = {k: v for k, v in ohe_prefixes.items() if k in df_work.columns}
print('Variables a OHE-encodear:', list(ohe_ok_current.keys()))

cols_antes = df_work.shape[1]

for col, prefix in ohe_ok_current.items():
    dummies = pd.get_dummies(df_work[col], prefix=prefix, drop_first=True, dummy_na=False, dtype=int)
    df_work = pd.concat([df_work, dummies], axis=1)

cols_despues = df_work.shape[1]
cols_nuevas = [c for c in df_work.columns if any(
    c.startswith(prefix + '_') for prefix in ohe_ok_current.values()
)]
print(f'\nColumnas antes del OHE:  {cols_antes}')
print(f'Columnas después del OHE: {cols_despues}')
print(f'Columnas dummies creadas ({len(cols_nuevas)}): {cols_nuevas}')

Variables a OHE-encodear: ['C203', 'C310', 'C311', 'C312', 'C313', 'SEGURO1', 'C376', 'C377']

Columnas antes del OHE:  150
Columnas después del OHE: 193
Columnas dummies creadas (43): ['C203_parentesco_2.0', 'C203_parentesco_3.0', 'C203_parentesco_4.0', 'C203_parentesco_5.0', 'C203_parentesco_6.0', 'C203_parentesco_7.0', 'C203_parentesco_8.0', 'C203_parentesco_11.0', 'C310_cat_ocup_2.0', 'C310_cat_ocup_3.0', 'C310_cat_ocup_4.0', 'C310_cat_ocup_6.0', 'C310_cat_ocup_7.0', 'C310_cat_ocup_8.0', 'C311_tipo_entidad_2.0', 'C311_tipo_entidad_3.0', 'C311_tipo_entidad_4.0', 'C311_tipo_entidad_5.0', 'C312_registro_sunat_2.0', 'C312_registro_sunat_3.0', 'C313_contabilidad_2.0', 'seguro_salud_2', 'seguro_salud_3', 'seguro_salud_4', 'seguro_salud_5', 'seguro_salud_6', 'C376_lengua_2', 'C376_lengua_3', 'C376_lengua_5', 'C376_lengua_9', 'C376_lengua_10', 'C376_lengua_11', 'C376_lengua_12', 'C376_lengua_13', 'C376_lengua_14', 'C377_etnia_2', 'C377_etnia_3', 'C377_etnia_4', 'C377_etnia_5', 'C377_etnia_

## 8. Construcción del dataset final codificado

Se seleccionan únicamente las columnas que serán insumo del modelamiento:  
- Variables numéricas útiles.  
- Variables binarias recodificadas (`_bin`).  
- Variables ordinales creadas.  
- Dummies del OHE.  
- Target `target_subempleo_horas`.

Se excluyen: leakage, identificadores, columnas originales ya codificadas.

In [9]:
# ── Columnas numéricas útiles ──────────────────────────────────────────────────
keep_num = [c for c in ['C208', 'C318_T', 'whoraT', 'INGTOT', 'INGTOTP', 'ingtrabw']
            if c in df_work.columns]

# ── Variables binarias recodificadas ─────────────────────────────────────────
keep_bin = [f'{c}_bin' for c in bin_ok_current if f'{c}_bin' in df_work.columns]

# ── Variables ordinales ───────────────────────────────────────────────────────
keep_ord = [c for c in ['nivel_educativo_ord', 'tamano_empresa_ord'] if c in df_work.columns]

# ── Dummies del OHE (solo columnas con prefijo_valor, no originales) ─────────
keep_ohe = [c for c in df_work.columns
            if any(c.startswith(prefix + '_') for prefix in ohe_ok_current.values())]

# ── Target ───────────────────────────────────────────────────────────────────
keep_target = ['target_subempleo_horas']

all_keep = keep_num + keep_bin + keep_ord + keep_ohe + keep_target

df_encoded = df_work[all_keep].copy()

print(f'Columnas en dataset final: {df_encoded.shape[1]}')
print(f'  Numéricas:   {len(keep_num)}  → {keep_num}')
print(f'  Binarias:    {len(keep_bin)}')
print(f'  Ordinales:   {len(keep_ord)}  → {keep_ord}')
print(f'  OHE dummies: {len(keep_ohe)}')
print(f'  Target:      1')

Columnas en dataset final: 64
  Numéricas:   6  → ['C208', 'C318_T', 'whoraT', 'INGTOT', 'INGTOTP', 'ingtrabw']
  Binarias:    12
  Ordinales:   2  → ['nivel_educativo_ord', 'tamano_empresa_ord']
  OHE dummies: 43
  Target:      1


## 9. Validación final

In [10]:
# ── Verificaciones de integridad ──────────────────────────────────────────────
assert 'target_subempleo_horas' in df_encoded.columns, 'ERROR: target no encontrado'
for v in ['P209H', 'C333', 'C334']:
    assert v not in df_encoded.columns, f'ERROR: variable de leakage {v} está en el dataset final'
for v in id_variables:
    assert v not in df_encoded.columns, f'ERROR: variable administrativa {v} está en el dataset final'

print('✓ target_subempleo_horas presente')
print('✓ Sin variables de leakage (P209H, C333, C334)')
print('✓ Sin identificadores administrativos')

# ── Tipos de datos ────────────────────────────────────────────────────────────
print('\nDtypes del dataset codificado:')
print(df_encoded.dtypes.value_counts().to_string())
non_numeric = df_encoded.select_dtypes(exclude=['number', 'bool']).columns.tolist()
if non_numeric:
    print(f'\n⚠ Columnas no numéricas: {non_numeric}')
else:
    print('\n✓ Todas las columnas son numéricas o booleanas')

# ── Nulos residuales ──────────────────────────────────────────────────────────
total_nulls = df_encoded.isna().sum()
cols_con_nulos = total_nulls[total_nulls > 0].sort_values(ascending=False)
print(f'\nColumnas con nulos residuales ({len(cols_con_nulos)}):')
print(cols_con_nulos.to_string() if len(cols_con_nulos) > 0 else 'Ninguna')

df_encoded.head(3)

✓ target_subempleo_horas presente
✓ Sin variables de leakage (P209H, C333, C334)
✓ Sin identificadores administrativos

Dtypes del dataset codificado:
int64      57
float64     7

✓ Todas las columnas son numéricas o booleanas

Columnas con nulos residuales (5):
INGTOT      774
ingtrabw    774
INGTOTP     774
C335_bin    744
C318_T        1


,C208,C318_T,whoraT,INGTOT,INGTOTP,ingtrabw,C207_bin,C361_1_bin,C361_5_bin,C364_1_bin,...,C376_lengua_14,C377_etnia_2,C377_etnia_3,C377_etnia_4,C377_etnia_5,C377_etnia_6,C377_etnia_7,C377_etnia_8,C377_etnia_9,target_subempleo_horas
0,60,36.0,36,1367.0,1367.0,1367.0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
1,49,30.0,30,1534.0,1534.0,1534.0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
2,52,40.0,40,1400.0,1400.0,1400.0,1,0,1,1,...,0,0,0,0,0,0,1,0,0,0


## 10. Guardar dataset codificado

In [11]:
DATA_PROC.mkdir(parents=True, exist_ok=True)

# ── Dataset codificado ────────────────────────────────────────────────────────
out_encoded = DATA_PROC / 'epen_encoded.csv'
df_encoded.to_csv(out_encoded, index=False, encoding='utf-8-sig')
print(f'Dataset codificado guardado: {out_encoded}')
print(f'  {df_encoded.shape[0]:,} filas × {df_encoded.shape[1]} columnas')

# ── Lista de columnas ─────────────────────────────────────────────────────────
out_cols = DATA_PROC / 'encoded_columns.csv'
pd.DataFrame({'columna': df_encoded.columns}).to_csv(out_cols, index=False, encoding='utf-8-sig')
print(f'Lista de columnas guardada: {out_cols}')

Dataset codificado guardado: g:\Mi unidad\UP - Ingeniería de la información\Semestre IX\Machine Learning\TrabajoFinal\clonProyecto\ML_PROYECTO_26_1\ml_project\data\processed\epen_encoded.csv
  24,054 filas × 64 columnas
Lista de columnas guardada: g:\Mi unidad\UP - Ingeniería de la información\Semestre IX\Machine Learning\TrabajoFinal\clonProyecto\ML_PROYECTO_26_1\ml_project\data\processed\encoded_columns.csv


## 11. Conclusiones

- Se codificaron las variables categóricas nominales del EPEN 2024 mediante **One-Hot Encoding** (`pd.get_dummies`, `drop_first=True`): `C203`, `C310`, `C311`, `C312`, `C313`, `SEGURO1`, `C376`, `C377`.
- Se conservaron como **variables ordinales** el nivel educativo (`C366` → `nivel_educativo_ord`) y el tamaño de empresa (`C317` → `tamano_empresa_ord`), dado que poseen un orden natural.
- Se recodificaron las **variables binarias** (1=Sí / 2=No → 1/0): `C207`, `C361_1`, `C361_5`, `C364_1`, `C364_2`, `C375_1`–`C375_6`, `C335`.
- Se excluyeron las variables con **riesgo de data leakage** (`P209H`, `C333`, `C334`) y los identificadores administrativos.
- El archivo `epen_encoded.csv` queda listo como insumo para la siguiente etapa: manejo de valores faltantes, división train/test y modelamiento.